# 05 — Agentic AI Credit Risk End-to-End V9.3 (Qwen Narrator)

```text
Borrower documents
      ↓
Deterministic extraction + feature engineering
      ↓
Qwen Agent
 ├─ predict_pd
 ├─ predict_ews
 ├─ predict_lgd
 ├─ predict_pd_cluster
 └─ query_credit_policy
      ↓
Python trusted ML binding + real Policy RAG
      ↓
Verified ML results + policy citations
      ↓
Qwen Credit Risk Narrator
      ↓
Holistic Credit Risk Assessment
```

### V9.3 changes
- Qwen is now the official final narrator.
- SahabatAI is removed from the active architecture.
- Legacy `SAHABAT_MODEL` is retained only as a compatibility alias.
- Narrator prompt is stricter about causal claims, out-of-range inputs,
  scale mismatch, KBLI interpretation, missing features, and policy claims.
- DeepEval narrator metric now evaluates factual grounding only;
  feature completeness is reported deterministically and does not automatically
  count as a groundedness failure.


In [ ]:
# ============================================================
# 0. RUNTIME CHECK
# ============================================================

import os, sys, subprocess, platform
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("IN_COLAB:", IN_COLAB)
print("Python   :", sys.version.split()[0])
print("Platform :", platform.platform())

try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
    ))
except Exception:
    print("GPU tidak terdeteksi. Aktifkan Runtime → Change runtime type → GPU.")


## 1. Clone source repository

In [ ]:
# ============================================================
# 1. CLONE REPOSITORY
# ============================================================

import os, subprocess
from pathlib import Path

REPO_URL = os.getenv("REPO_URL", "https://github.com/fbellaa/odp-bni-capstone.git")
BRANCH = os.getenv("REPO_BRANCH", "feat/ds-modeling")
REPO_DIR = Path("/content/odp-bni-capstone")

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    print("Repo sudah ada:", REPO_DIR)

os.environ["ODP_REPO_ROOT"] = str(REPO_DIR)
os.environ["ML_ARTIFACT_ROOT"] = str(REPO_DIR / "ml" / "artifacts")
print("REPO_DIR:", REPO_DIR)


## 2. Overlay Agentic AI V9.3 + RAG

In [ ]:
# ============================================================
# 2. OVERLAY V9.3 PACKAGE + POLICY RAG
# ============================================================

from pathlib import Path
import shutil
import zipfile

agent_init = (
    REPO_DIR
    / "ml"
    / "agentic_ai"
    / "__init__.py"
)

needs_overlay = True


# ============================================================
# CHECK INSTALLED VERSION
# ============================================================

if agent_init.exists():

    text = agent_init.read_text(
        errors="ignore"
    )

    needs_overlay = (
        '__version__ = "9.3.0"'
        not in text
    )


# ============================================================
# OVERLAY PACKAGE
# ============================================================

if needs_overlay:

    if not IN_COLAB:

        raise RuntimeError(
            "Copy package V9.3 ke repository secara manual."
        )


    from google.colab import files


    print(
        "Upload "
        "agentic_ai_ml_package_v9_3_qwen_narrator.zip"
    )


    uploaded = files.upload()


    zip_names = [

        name

        for name
        in uploaded

        if name.lower().endswith(
            ".zip"
        )
    ]


    if not zip_names:

        raise RuntimeError(
            "Tidak ada ZIP yang di-upload."
        )


    # ========================================================
    # SAVE UPLOADED ZIP
    # ========================================================

    zip_name = zip_names[0]

    zpath = (
        Path("/content")
        / zip_name
    )

    zpath.write_bytes(
        uploaded[
            zip_name
        ]
    )


    # ========================================================
    # EXTRACT
    # ========================================================

    temp = Path(
        "/content/v9_3_overlay"
    )

    shutil.rmtree(
        temp,
        ignore_errors=True,
    )

    temp.mkdir(
        parents=True,
        exist_ok=True,
    )


    with zipfile.ZipFile(
        zpath
    ) as zf:

        zf.extractall(
            temp
        )


    # ========================================================
    # 1. COPY ml/agentic_ai
    # ========================================================

    agent_candidates = list(
        temp.rglob(
            "ml/agentic_ai"
        )
    )


    if not agent_candidates:

        raise RuntimeError(
            "ZIP tidak berisi "
            "ml/agentic_ai."
        )


    target_agent = (
        REPO_DIR
        / "ml"
        / "agentic_ai"
    )


    shutil.copytree(
        agent_candidates[0],
        target_agent,
        dirs_exist_ok=True,
    )


    print(
        "✓ ml/agentic_ai V9.3 overlay selesai"
    )


    # ========================================================
    # 2. COPY RAG FILES
    #
    # Penting:
    # V9 memakai copilot/rag dari package.
    # Jangan copy seluruh folder copilot karena repository
    # sudah punya konfigurasi, llm client, dokumen, dll.
    # ========================================================

    rag_candidates = list(
        temp.rglob(
            "copilot/rag"
        )
    )


    if rag_candidates:

        target_rag = (
            REPO_DIR
            / "copilot"
            / "rag"
        )


        target_rag.mkdir(
            parents=True,
            exist_ok=True,
        )


        shutil.copytree(
            rag_candidates[0],
            target_rag,
            dirs_exist_ok=True,
        )


        print(
            "✓ copilot/rag overlay selesai"
        )

    else:

        raise RuntimeError(
            "ZIP V9.3 tidak berisi "
            "copilot/rag."
        )


    # ========================================================
    # 3. COPY AGENTIC REQUIREMENTS
    # ========================================================

    reqs = list(
        temp.rglob(
            "requirements-agentic-ai.txt"
        )
    )


    if reqs:

        shutil.copy2(
            reqs[0],
            REPO_DIR
            / "requirements-agentic-ai.txt",
        )

        print(
            "✓ requirements-agentic-ai.txt updated"
        )


    # ========================================================
    # 4. COPY RAG COMPATIBILITY MODULES IF REPO DOES NOT HAVE THEM
    # ========================================================

    compatibility_files = {
        "copilot/konfigurasi.py": REPO_DIR / "copilot" / "konfigurasi.py",
        "copilot/llm/klien.py": REPO_DIR / "copilot" / "llm" / "klien.py",
        "copilot/dokumen/pdf.py": REPO_DIR / "copilot" / "dokumen" / "pdf.py",
    }

    for relative_source, target_file in compatibility_files.items():

        candidates = list(
            temp.rglob(relative_source)
        )

        if not candidates:
            continue

        # Preserve an existing repository implementation.
        if target_file.exists():
            print(
                f"✓ keep existing {target_file.relative_to(REPO_DIR)}"
            )
            continue

        target_file.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        shutil.copy2(
            candidates[0],
            target_file,
        )

        print(
            f"✓ added {target_file.relative_to(REPO_DIR)}"
        )


    # ========================================================
    # FINAL VERSION CHECK
    # ========================================================

    installed_text = (
        agent_init
        .read_text(
            errors="ignore"
        )
    )


    if (
        '__version__ = "9.3.0"'
        not in installed_text
    ):

        raise RuntimeError(
            "Overlay selesai tetapi "
            "versi V9.3 tidak terdeteksi."
        )


    print(
        "\n✅ V9.3 + Policy RAG overlay selesai"
    )


else:

    print(
        "✓ ml/agentic_ai V9.3 "
        "sudah tersedia"
    )


    # Tetap cek RAG folder
    rag_dir = (
        REPO_DIR
        / "copilot"
        / "rag"
    )


    if rag_dir.exists():

        print(
            "✓ copilot/rag tersedia"
        )

    else:

        print(
            "⚠ ml/agentic_ai sudah V9.3 "
            "tetapi copilot/rag belum ada."
        )


## 3. Upload ML artifacts dari laptop

In [ ]:
# ============================================================
# 3. UPLOAD LOCAL ML ARTIFACTS
# ============================================================

from pathlib import Path
import shutil, zipfile
from google.colab import files

TARGET_ARTIFACT_DIR = REPO_DIR / "ml" / "artifacts"

print("Upload artifacts.zip")
uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Upload tepat satu ZIP. Terdeteksi: {zip_names}")

zpath = Path("/content") / zip_names[0]
zpath.write_bytes(uploaded[zip_names[0]])
extract_dir = Path("/content/artifact_upload")
shutil.rmtree(extract_dir, ignore_errors=True)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract_dir)

candidates = []
for p0 in extract_dir.rglob("artifacts"):
    if p0.is_dir():
        dirs = {x.name for x in p0.iterdir() if x.is_dir()}
        if {"pd", "ews", "lgd", "pd_cluster"} & dirs:
            candidates.append(p0)

if not candidates:
    raise FileNotFoundError("Folder artifacts tidak ditemukan di ZIP.")

source = sorted(candidates, key=lambda x: len(x.parts))[0]
shutil.rmtree(TARGET_ARTIFACT_DIR, ignore_errors=True)
TARGET_ARTIFACT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, TARGET_ARTIFACT_DIR)

for key in ["pd", "ews", "lgd", "pd_cluster"]:
    model_dir = TARGET_ARTIFACT_DIR / key
    print(f"\n[{key}]")
    if model_dir.exists():
        for f in sorted(model_dir.iterdir()):
            print(" -", f.name)
    else:
        print(" !! missing folder")


## 4. Install runtime dependencies

In [ ]:
# ============================================================
# 4. INSTALL DEPENDENCIES
# ============================================================

import sys, subprocess
from pathlib import Path

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run([
    "apt-get", "install", "-y", "-qq",
    "tesseract-ocr", "tesseract-ocr-ind", "tesseract-ocr-eng",
    "zstd", "curl"
], check=True)

ai_req = REPO_DIR / "requirements-agentic-ai.txt"
if not ai_req.exists():
    raise FileNotFoundError(ai_req)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ai_req)], check=True)

# Python 3.13 compatible runtime. XGBoost is kept close to the LGD training artifact.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "joblib", "scikit-learn", "pandas", "numpy", "scipy", "xgboost==2.1.4"
], check=True)

print("✓ Dependencies installed")


## 5. Install + start Ollama

In [ ]:
# ============================================================
# 5. INSTALL + START OLLAMA
# ============================================================

import os, shutil, subprocess, time, requests
from pathlib import Path

if shutil.which("ollama") is None:
    archive = Path("/tmp/ollama-linux-amd64.tar.zst")
    subprocess.run([
        "curl", "-fL", "--retry", "5", "--retry-delay", "3",
        "-o", str(archive),
        "https://ollama.com/download/ollama-linux-amd64.tar.zst"
    ], check=True)
    subprocess.run(["tar", "--zstd", "-xf", str(archive), "-C", "/usr"], check=True)

ollama_bin = shutil.which("ollama") or "/usr/bin/ollama"
subprocess.run(["pkill", "-f", "ollama serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

env = os.environ.copy()
env["OLLAMA_HOST"] = "127.0.0.1:11434"
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_GPU_OVERHEAD"] = "536870912"
env["OLLAMA_MODELS"] = "/content/ollama_models"
Path(env["OLLAMA_MODELS"]).mkdir(parents=True, exist_ok=True)

log = open("/tmp/ollama.log", "w")
proc = subprocess.Popen([ollama_bin, "serve"], env=env, stdout=log, stderr=subprocess.STDOUT)

for i in range(90):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.ok:
            print("✓ Ollama ready")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(Path("/tmp/ollama.log").read_text(errors="ignore")[-5000:])
    raise RuntimeError("Ollama gagal start")


## 6. Model configuration

In [ ]:
# ============================================================
# 6. MODEL CONFIG + PULL — QWEN AGENT + QWEN NARRATOR
# ============================================================

import os
import subprocess

QWEN_AGENT_MODEL = "qwen2.5:7b-instruct"

# Optional semantic extraction fallback uses the same Qwen family.
QWEN_EXTRACTOR_MODEL = QWEN_AGENT_MODEL

# Final narrator is officially Qwen in V9.3.
QWEN_NARRATOR_MODEL = QWEN_AGENT_MODEL

# Only needed for scanned/image pages.
VLM_MODEL = "qwen3-vl:4b-instruct"

os.environ["QWEN_AGENT_MODEL"] = QWEN_AGENT_MODEL
os.environ["QWEN_EXTRACTOR_MODEL"] = QWEN_EXTRACTOR_MODEL
os.environ["QWEN_NARRATOR_MODEL"] = QWEN_NARRATOR_MODEL
os.environ["VLM_MODEL"] = VLM_MODEL

# Legacy alias so older imported code remains harmless.
os.environ["SAHABAT_MODEL"] = QWEN_NARRATOR_MODEL

os.environ["AI_QWEN_SEMANTIC_FALLBACK"] = "0"
os.environ["AI_TIMEOUT"] = "300"
os.environ["AI_OLLAMA_KEEP_ALIVE"] = "5m"
os.environ["AI_NARRATOR_TEMPERATURE"] = "0.1"

models_to_pull = list(dict.fromkeys([
    QWEN_AGENT_MODEL,
    QWEN_NARRATOR_MODEL,
    "nomic-embed-text",
]))

for model in models_to_pull:
    print("\nPulling/checking:", model)
    subprocess.run(
        ["ollama", "pull", model],
        check=True,
    )

print("\nQwen agent    :", QWEN_AGENT_MODEL)
print("Qwen narrator :", QWEN_NARRATOR_MODEL)
print("VLM optional  :", VLM_MODEL)


## 7. Initialize V9.3 + inspect feature contracts

In [ ]:
# ============================================================
# 7. INITIALIZE V9.3
# ============================================================

import os, sys
from pathlib import Path

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

for name in list(sys.modules):
    if name == "ml.agentic_ai" or name.startswith("ml.agentic_ai."):
        del sys.modules[name]

from ml.agentic_ai import CreditRiskAIPipeline, __version__
from ml.agentic_ai.config import SETTINGS

print("Agentic AI version:", __version__)
print("Qwen agent          :", SETTINGS.qwen_agent_model)
print("Semantic fallback   :", SETTINGS.use_qwen_semantic_fallback)
print("VLM                 :", SETTINGS.vlm_model)
print("Qwen narrator       :", SETTINGS.qwen_narrator_model)

ai = CreditRiskAIPipeline()
store = ai.store

print("\nFEATURE CONTRACTS")
for key in ["pd", "ews", "lgd", "pd_cluster"]:
    defs = store.feature_defs(key)
    print(f"{key.upper():<10}: {len(defs)} features | first={[f.name for f in defs[:8]]}")


In [ ]:
# ============================================================
# 8. CHAMPION LOAD SMOKE TEST
# ============================================================

load_status = {}
for key in ["pd", "ews", "lgd", "pd_cluster"]:
    print("\n" + "=" * 70)
    print(key.upper())
    try:
        bundle = store.bundle(key)
        load_status[key] = "OK"
        print("✓ champion load OK | type:", type(bundle))
        if key == "pd_cluster" and isinstance(bundle, dict):
            print("cluster bundle fields:", sorted(bundle.keys()))
    except Exception as e:
        load_status[key] = f"{type(e).__name__}: {e}"
        print("❌ load failed:", load_status[key])

print("\nSummary:", load_status)


## 9. Upload all documents for one borrower

In [ ]:
# ============================================================
# 9. MULTI-DOCUMENT TEXT EXTRACTION
# ============================================================

from google.colab import files
from ml.agentic_ai.document_extraction import extract_documents_multimodal

print("Upload SEMUA dokumen untuk satu debitur/case:")
uploaded_docs = files.upload()
if not uploaded_docs:
    raise RuntimeError("Tidak ada dokumen yang di-upload.")

document_inputs = list(uploaded_docs.items())

docs = extract_documents_multimodal(
    document_inputs,
    ollama_url="http://127.0.0.1:11434",
    vlm_model=VLM_MODEL,
    min_native_chars=80,
    min_ocr_chars=80,
)

print("\nSources:", docs.source_names)
print("Warnings:", docs.warnings or "None")
print("\nExtraction method per page:")
for p0 in docs.pages:
    print(f"- {p0.source_name} | page={p0.page} | method={p0.method} | chars={len(p0.text)}")


## 10. Deterministic document → raw-fact mapping

In [ ]:
# ============================================================
# 10. DETERMINISTIC DOCUMENT MAPPING
# ============================================================

# No LLM mapping here. This should be fast.
extraction, llm_text, reduction, strategy = ai.extract_from_documents(
    docs,
    manual_input=None,
    use_qwen_semantic_fallback=False,
)

print("Extraction strategy:", strategy)
print("Borrower           :", extraction.borrower_name)
print("Raw facts          :", len(extraction.raw_facts))

print("\nMAPPED RAW FACTS")
for name, item in extraction.raw_facts.items():
    method = item.evidence.extraction_method if item.evidence else None
    src = item.evidence.source_document if item.evidence else None
    page = item.evidence.page if item.evidence else None
    print(f"- {name:<28} = {item.value} {item.unit or ''} | {method} | {src} p.{page}")


## 11. Optional RM supplementary input

In [ ]:
# ============================================================
# 11. OPTIONAL MANUAL RM INPUT
# ============================================================

# Fill only information that is genuinely known and not in the uploaded documents.
manual_input = {
    # "app_rating_internal": "BBB",
    # "app_skor_kredit": 72,
    # "app_sektor_kbli": "41011",
    # "app_jenis_fasilitas": "KMK",
    # "app_tenor_bulan": 12,
    # "app_plafon_diminta_rp": 50_000_000_000,
    # "perilaku_dpd": 0,
    # "perilaku_kolektibilitas": 1,
}

print("Manual fields:", manual_input)


## 12. Deterministic feature engineering + coverage

In [ ]:
# ============================================================
# 12. BUILD MODEL FEATURES
# ============================================================

feature_context = ai.feature_engineer.build(
    extraction,
    manual_input=manual_input,
)

for key in ["pd", "ews", "lgd", "pd_cluster"]:
    ctx = feature_context[key]
    print("\n" + "=" * 70)
    print(key.upper())
    print("Expected     :", ctx["expected_feature_count"])
    print("Observed     :", ctx["observed_feature_count"])
    print("Missing      :", len(ctx["missing_feature_names"]))
    print("Completeness :", ctx["feature_completeness_percent"], "%")
    print("Observed features:")
    for name, value in ctx["features"].items():
        provenance = ctx["feature_provenance"].get(name, {})
        print(f" - {name}: {value} | {provenance.get('source')} | {provenance.get('formula')}")


## 13. Qwen mandatory ML tool calling

## Policy RAG setup

V9.3 memakai RAG kebijakan yang kamu berikan sebagai **tool Qwen**.

File RAG:
- `copilot/rag/indeks.py`
- `copilot/rag/pencarian.py`
- `copilot/rag/potong.py`

RAG membutuhkan:
1. PDF kebijakan di lokasi yang dipakai `copilot.konfigurasi.POLICY_DIR`,
2. embedding model sesuai `PENGATURAN.model_untuk("embedding")`,
3. index kebijakan yang sudah dibangun.

Qwen akan memanggil `query_credit_policy` setelah 4 ML tools dicoba.


In [ ]:
# ============================================================
# POLICY RAG — PREPARE REAL INDEX
# ============================================================

import os
from google.colab import files

os.environ["AI_RAG_ENABLED"] = "1"
os.environ["AI_RAG_MOCK"] = "0"
os.environ["RAG_EMBED_MODEL"] = "nomic-embed-text"

from copilot.konfigurasi import POLICY_DIR
from copilot.rag.indeks import bangun_index, index_tersedia
from ml.agentic_ai.rag_adapter import rag_preflight

POLICY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

policy_files = sorted(
    POLICY_DIR.glob("*.pdf")
)

if not policy_files:
    print("Upload PDF kebijakan / regulasi:")
    uploaded_policy = files.upload()

    for filename, data in uploaded_policy.items():
        if filename.lower().endswith(".pdf"):
            target = POLICY_DIR / filename
            target.write_bytes(data)
            print("✓ Saved:", target)

    policy_files = sorted(
        POLICY_DIR.glob("*.pdf")
    )

if not policy_files:
    raise RuntimeError(
        "Tidak ada PDF policy."
    )

print("\nPolicy PDFs:")
for path in policy_files:
    print(" -", path.name)

if not index_tersedia():
    print("\nBuilding Policy RAG index...")
    idx = bangun_index(
        paksa=True,
    )
    print(
        "✓ Index built:",
        len(idx),
        "chunks",
    )
else:
    print("\n✓ Existing RAG index reused")

rag_status = rag_preflight()

print("\nRAG PREFLIGHT")
print("RAG importable      :", rag_status["rag_importable"])
print("RAG index available :", rag_status["index_available"])
print("Embedding model     :", rag_status["embedding_model"])
print("RAG error           :", rag_status["error"])

if not rag_status["index_available"]:
    raise RuntimeError(
        "Policy RAG belum ready."
    )


In [ ]:
# ============================================================
# POLICY RAG SMOKE TEST
# ============================================================

from ml.agentic_ai.rag_adapter import query_credit_policy

rag_smoke_result = query_credit_policy(
    "Apa faktor yang harus dinilai dalam penetapan kualitas kredit?",
    top_k=3,
)

print("RAG status       :", rag_smoke_result["status"])
print("Citation count   :", rag_smoke_result["citation_count"])
print("Synthesis warning:", rag_smoke_result.get("synthesis_warning"))

for citation in rag_smoke_result.get("citations", [])[:3]:
    print(" -", citation)


In [ ]:
# ============================================================
# 13. RUN QWEN AGENT — 4 ML TOOLS + REAL POLICY RAG
# ============================================================

import json

print("=" * 80)
print("RUNNING QWEN AGENT")
print("=" * 80)

qwen_context = ai._qwen_context(
    feature_context
)

policy_context = ai._policy_context(
    extraction
)

print(
    "Policy context:",
    json.dumps(
        policy_context,
        indent=2,
        ensure_ascii=False,
        default=str,
    ),
)

agent_result = ai.agent.run(
    qwen_context,
    policy_context=policy_context,
    execute_tools=True,
    use_rag=True,
)

print("\n" + "=" * 80)
print("AGENT EXECUTION COMPLETE")
print("=" * 80)

print("ML tool coverage           :", agent_result.record.coverage)
print("Qwen ML tool coverage      :", agent_result.record.qwen_coverage)
print("Qwen overall agent coverage:", agent_result.record.qwen_agent_tool_coverage)
print("RAG attempted by Qwen      :", agent_result.record.rag_qwen_attempted)
print("RAG retrieved              :", agent_result.record.rag_retrieved)
print("Stopped reason             :", agent_result.stopped_reason)


In [ ]:
# ============================================================
# QWEN TOOL CALLING AUDIT — ML + POLICY RAG
# ============================================================

print("ML tool coverage          :", agent_result.record.coverage)
print("Qwen ML tool coverage     :", agent_result.record.qwen_coverage)
print(
    "Qwen overall agent coverage:",
    agent_result.record.qwen_agent_tool_coverage,
)
print("RAG attempted by Qwen     :", agent_result.record.rag_qwen_attempted)
print("RAG retrieved             :", agent_result.record.rag_retrieved)
print("Stopped reason            :", agent_result.stopped_reason)

print("\nTOOL TRACE AUDIT")

for t in agent_result.record.traces:

    print(
        f"- {t.name:<22}"
        f" caller={t.caller:<15}"
        f" success={t.success}"
        f" duration_ms={t.duration_ms}"
        f" binding={t.binding_source}"
    )

    if t.name.startswith("predict_"):
        print(
            "  bound_feature_count:",
            len((t.arguments or {}).get("features", {})),
        )

    if t.name == "query_credit_policy":
        print(
            "  RAG query:",
            (t.arguments or {}).get("query"),
        )
        if t.result:
            print(
                "  RAG status:",
                t.result.get("status"),
            )
            print(
                "  citations:",
                t.result.get("citation_count", 0),
            )

    if t.error:
        print("  error:", t.error)

print("\nPOLICY RAG RESULT")
print(
    json.dumps(
        agent_result.record.rag_result,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)


## 14. Verified ML results

In [ ]:
# ============================================================
# 14. VERIFIED ML RESULTS
# ============================================================

verified = agent_result.record.last_success_by_name

for tool in ["predict_pd", "predict_ews", "predict_lgd", "predict_pd_cluster"]:
    print("\n" + "=" * 70)
    print(tool)
    if tool in verified:
        print("✓ SUCCESS")
        print(verified[tool])
    else:
        print("❌ NO SUCCESSFUL RESULT")
        errors = [t.error for t in agent_result.record.traces if t.name == tool and t.error]
        print("Errors:", errors)


## 15. Final Indonesian assessment

In [ ]:
# ============================================================
# 15. QWEN CREDIT RISK NARRATOR
# ============================================================

answer = ai.narrator.narrate(
    extraction=extraction,
    feature_context=feature_context,
    agent=agent_result,
    document_warnings=docs.warnings,
)

print(answer)


## 16. Optional semantic fallback for unfamiliar document wording

In [ ]:
# ============================================================
# OPTIONAL ONLY — QWEN SEMANTIC FALLBACK
# ============================================================
# Use this only when deterministic mapping misses an unfamiliar document label.
# Deterministic values remain authoritative and are never overwritten by Qwen.

RUN_QWEN_SEMANTIC_FALLBACK = False

if RUN_QWEN_SEMANTIC_FALLBACK:
    extraction_fallback, llm_text, reduction, strategy = ai.extract_from_documents(
        docs,
        manual_input=manual_input,
        use_qwen_semantic_fallback=True,
    )
    print("Strategy:", strategy)
    print("Raw facts:", len(extraction_fallback.raw_facts))
    for k, v in extraction_fallback.raw_facts.items():
        print("-", k, "=", v.value)


## 17. Evaluation — DeepEval (V9)

Evaluasi dibuat **per layer** agar jelas apa yang sedang diuji:

1. **Document Mapper** — apakah field dari dokumen terbaca benar.
2. **Qwen Tool Calling** — apakah Qwen benar-benar memanggil 4 ML tools dengan argument yang tepat.
3. **Qwen Narrator** — apakah narasi akhir grounded pada verified ML results.
4. **End-to-End Scorecard** — berapa model yang sukses dan bagaimana kualitas inputnya.

> Untuk kasus LGD artifact yang sedang error, **tool-calling Qwen tetap bisa mendapat skor penuh** apabila Qwen sudah memanggil `predict_lgd` dengan benar. Runtime failure LGD dicatat terpisah sebagai model-runtime issue.


In [ ]:
# ============================================================
# 17A. IMPORT V9.3 EVALUATION HELPERS
# ============================================================

from ml.agentic_ai.eval.deepeval_v9 import (
    evaluate_document_mapper,
    evaluate_qwen_tool_calling,
    evaluate_narrator_groundedness,
    evaluate_feature_completeness,
    evaluate_end_to_end,
    save_evaluation_report,
)

print("✓ V9.3 evaluation helpers loaded")

### 17B. Layer 1 — Document Mapping Accuracy

Golden dataset bawaan berisi contoh financial statement, application fields, EWS behavior, cash flow, dan irrelevant text.

Ini **deterministic exact-match evaluation**, karena angka seperti total aset lebih tepat diuji terhadap ground truth daripada dinilai LLM.


In [ ]:
# ============================================================
# 17B. DOCUMENT MAPPER EVALUATION
# ============================================================

mapper_eval = evaluate_document_mapper(
    verbose=True,
)

print("\nMAPPER SUMMARY")
print("Field Recall    :", mapper_eval["field_recall"])
print("Field Precision :", mapper_eval["field_precision"])
print("Field F1        :", mapper_eval["field_f1"])
print("Forbidden hits  :", mapper_eval["forbidden_hallucination_hits"])

### 17C. Layer 2 — Qwen Tool Calling with DeepEval

DeepEval memeriksa **Qwen-originated calls saja**, bukan Python fallback.

Target:
- `Qwen tool coverage = 1.0`
- exact four-tool set
- exact argument fidelity
- `DeepEval ToolCorrectness = 1.0`

Kalau LGD model runtime error tetapi Qwen sudah memanggil tool LGD dengan argument benar, **tool-calling score tetap dapat 1.0**.


In [ ]:
# ============================================================
# 17C. QWEN TOOL SELECTION + TRUSTED BINDING DEEPEVAL
# ============================================================

tool_eval = evaluate_qwen_tool_calling(
    agent_result=agent_result,
    feature_context=feature_context,
    verbose=True,
)

print("\nInterpretation:")
print(
    "- Qwen selects tool names; it no longer transports ML feature values."
)
print(
    "- Python binding fidelity verifies the exact deterministic "
    "feature_context reached the selected ML tool."
)


### 17D. Layer 3 — Qwen Narrator Groundedness

DeepEval `GEval` menggunakan local Ollama model sebagai judge.

Default judge = Qwen agent model. Untuk benchmark yang lebih kuat, isi model judge lain yang tersedia di Ollama:

```python
DEEPEVAL_JUDGE_MODEL = "nama-model-judge"
```

Judge memeriksa apakah narrator:
- tidak mengarang angka,
- tidak menyatakan model gagal sebagai sukses,
- tidak menyatakan nilai imputed sebagai data observasi,
- tidak membuat causal claim yang tidak didukung.


In [ ]:
# ============================================================
# 17D. QWEN CREDIT RISK NARRATOR DEEPEVAL
# ============================================================

import os

# Opsional: ganti dengan local judge lain di Ollama.
# Contoh:
# os.environ["DEEPEVAL_JUDGE_MODEL"] = "qwen2.5:7b-instruct"

narrator_eval = evaluate_narrator_groundedness(
    answer=answer,
    agent_result=agent_result,
    feature_context=feature_context,
    judge_model=os.getenv("DEEPEVAL_JUDGE_MODEL") or None,
    verbose=True,
)

In [ ]:
# ============================================================
# 17D.2 INPUT COMPLETENESS — DETERMINISTIC, NOT LLM JUDGE
# ============================================================

input_completeness_eval = evaluate_feature_completeness(
    feature_context=feature_context,
    verbose=True,
)

print(
    "\nCatatan: completeness dilaporkan sebagai kualitas input; "
    "tidak otomatis diinterpretasikan sebagai akurasi model."
)


### 17E. End-to-End Evaluation Report

Tidak dibuat satu weighted score yang arbitrer. Report mempertahankan skor tiap layer agar saat presentasi bisa menjelaskan **di bagian mana sistem bagus atau masih bermasalah**.


In [ ]:
# ============================================================
# 17E. END-TO-END SCORECARD + SAVE JSON
# ============================================================

evaluation_report = evaluate_end_to_end(
    extraction=extraction,
    feature_context=feature_context,
    agent_result=agent_result,
    answer=answer,
    mapper_eval=mapper_eval,
    tool_eval=tool_eval,
    narrator_eval=narrator_eval,
)

print("=" * 80)
print("V9.3 END-TO-END EVALUATION")
print("=" * 80)

print("Borrower              :", evaluation_report["borrower_name"])
print("Raw facts             :", evaluation_report["raw_fact_count"])
print("Qwen tool coverage    :", evaluation_report["qwen_tool_coverage"])
print(
    "Successful ML models :",
    f'{evaluation_report["successful_model_count"]}/'
    f'{evaluation_report["required_model_count"]}'
)
print("Model success rate    :", evaluation_report["model_success_rate"])
print("Successful models     :", evaluation_report["successful_models"])
print("Failed models         :", evaluation_report["failed_models"])
print("Stopped reason        :", evaluation_report["agent_stopped_reason"])
print("Final answer nonempty :", evaluation_report["final_answer_nonempty"])

print("\nFeature completeness:")
for model_name, completeness in evaluation_report[
    "feature_completeness_percent"
].items():
    print(f" - {model_name:<12}: {completeness}%")

report_path = save_evaluation_report(
    evaluation_report,
    "/content/agentic_ai_v9_evaluation_report.json",
)

print("\n✓ Evaluation report saved:", report_path)

In [ ]:
# ============================================================
# 17F. OPTIONAL DOWNLOAD EVALUATION REPORT
# ============================================================

from google.colab import files

files.download(
    "/content/agentic_ai_v9_evaluation_report.json"
)